# Combo B: The Alien Biology Exam"Can an AI reason about unfamiliar biology?"Skin a system, scale difficulty, run agents on opaque tasks.

In [ ]:
import sys, os, mathsys.path.insert(0, os.path.join(os.getcwd(), "demos"))from _shared import make_disease_system, oracle_agent, random_agent, zero_agentfrom alienbio.bio import AgentInterface, DiagnoseTaskfrom alienbio.scenarios.skinning import generate_name_map, generate_descriptionfrom alienbio.scenarios.difficulty_curve import DifficultySpec, measure_difficulty_curvefrom alienbio.bio.comparison import AgentStats, ComparisonTablefrom alienbio.viz import difficulty_curve_plot, agent_comparison_chart

## Alien Biology

In [ ]:
system, baseline, perturbs = make_disease_system(seed=42)name_map = generate_name_map(system, seed=42)print(generate_description(system, detail_level=2, name_map=name_map, seed=42))

## Difficulty-Scaled Tasks

In [ ]:
spec = DifficultySpec(levels=[])for level, label, n_cand in [(1, "easy", 2), (2, "medium", 4), (3, "hard", 8)]:    tasks = [DiagnoseTask(perturbs[:n_cand] if n_cand <= len(perturbs) else perturbs,                          applied_index=i % min(n_cand, len(perturbs)))             for i in range(3)]    spec.add_level(level, label, tasks)

## Run Agents

In [ ]:
iface = AgentInterface(system)curves = []for name, fn in [("oracle", oracle_agent), ("random", random_agent), ("zero", zero_agent)]:    curves.append(measure_difficulty_curve(spec, iface, fn, agent_name=name))

In [ ]:
difficulty_curve_plot(curves, title="Alien Exam: Difficulty Curves")

## Leaderboard

In [ ]:
all_stats = []for curve in curves:    scores = [s for pt in curve.points for s in pt.scores]    n = len(scores) if scores else 1    mean = sum(scores)/n if scores else 0.0    var = sum((s-mean)**2 for s in scores)/n if scores else 0.0    pass_rate = sum(1 for s in scores if s >= 0.5)/n if scores else 0.0    all_stats.append(AgentStats(curve.agent_name, mean, math.sqrt(var),        min(scores) if scores else 0.0, max(scores) if scores else 0.0, n, pass_rate))table = ComparisonTable(agents=all_stats)for i, a in enumerate(table.ranking):    print(f"#{i+1} {a.agent_name}: {a.mean:.2f}")

In [ ]:
agent_comparison_chart(table, title="Alien Exam: Leaderboard")